# Solution A1 · Gradient boosting for regression from scratch: trees on the residuals

**Type**: Implement from scratch

## Objective

Implement gradient boosting with the squared loss using only `DecisionTreeRegressor` as a building block: constant initial prediction, small trees on the residuals, update with shrinkage. Check that it matches `GradientBoostingRegressor` in the first rounds, measure the error curve as a function of the
rounds and find the round at which the test error starts to get worse.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor


def generate():
    rng = np.random.default_rng(0)
    X = rng.uniform(-3, 3, (600, 3))
    y = np.sin(X[:, 0]) + 0.09 * X[:, 1] ** 2 + rng.normal(0, 0.3, 600)
    return train_test_split(X, y, test_size=0.4, random_state=0)


X_tr, X_te, y_tr, y_te = generate()

## Commented solution

In [ ]:
def fit_gb(X, y, n_rounds, eta, depth):
    f0 = y.mean()
    F = np.full(len(y), f0)
    trees = []
    for _ in range(n_rounds):
        tree = DecisionTreeRegressor(max_depth=depth, criterion="squared_error", random_state=0).fit(X, y - F)      # small tree on the residuals
        F = F + eta * tree.predict(X)
        trees.append(tree)
    return f0, trees


def predict_gb(model, X, eta):
    f0, trees = model
    return f0 + eta * sum(t.predict(X) for t in trees)


def error_curve(model, X, y, eta):
    f0, trees = model
    F = np.full(len(X), f0)
    curve = []
    for t in trees:
        F = F + eta * t.predict(X)
        curve.append(np.mean((y - F) ** 2))
    return curve

In [ ]:
model = fit_gb(X_tr, y_tr, 300, 0.3, 2)
train_curve = error_curve(model, X_tr, y_tr, 0.3)
test_curve = error_curve(model, X_te, y_te, 0.3)
best_round = int(np.argmin(test_curve)) + 1
best_mse = float(min(test_curve))
print(f"minimum test MSE {best_mse:.4f} at round {best_round} | after 300 rounds: test {test_curve[-1]:.4f}, training {train_curve[-1]:.4f}")

In [ ]:
def difference(B):
    mine = predict_gb(fit_gb(X_tr, y_tr, B, 0.1, 3), X_te, 0.1)
    sk = GradientBoostingRegressor(n_estimators=B, learning_rate=0.1, max_depth=3, criterion="squared_error", random_state=0).fit(X_tr, y_tr).predict(X_te)
    return float(np.abs(mine - sk).max())


diff_5_rounds, diff_100_rounds = difference(5), difference(100)
print(f"maximum difference from scikit-learn: after 5 rounds {diff_5_rounds:.1e} | after 100 rounds {diff_100_rounds:.3f}")

## Check

In [ ]:
with verify():
    check(len(model[1]) == 300 and close(model[0], y_tr.mean()), "300 trees and initial prediction equal to the mean", "f0 = y.mean()")
    check(len(train_curve) == 300 and all(np.diff(train_curve) <= 1e-9), "the training error never goes up", "each tree reduces (or leaves equal) the training error")
    check(close(test_curve[-1], np.mean((predict_gb(model, X_te, 0.3) - y_te) ** 2)) and close(train_curve[0], np.mean((y_tr - (y_tr.mean() + 0.3 * model[1][0].predict(X_tr))) ** 2)), "curves consistent with the full predictions", "MSE after each round")
    check(train_curve[-1] < 0.05 and test_curve[-1] > test_curve[best_round - 1] + 0.02 and 5 <= best_round <= 100, "the training error goes below 0.05; the test error has its minimum between round 5 and 100 and then climbs back by over 0.02", "overfitting: with eta=0.3 the test error climbs back")
    check(close(best_mse, min(test_curve)) and best_mse < 0.15, "minimum test MSE below 0.15", "min(test_curve)")
    check(diff_5_rounds < 1e-9 and diff_100_rounds < 0.25, "after 5 rounds the predictions match scikit-learn (1e-9); after 100 they differ by at most 0.25", "criterion='squared_error', random_state=0")
done()

## What you learned

- **Boosting is a four-line loop:** initial prediction (the mean), residuals, small tree, update with step `eta`. With 5 rounds the by-hand implementation matches scikit-learn's to 1e-9 (the predictions are identical to machine precision).
- **The training error always drops, the test error does not:** with `eta = 0.3` and trees of 4 leaves the training error gets below 0.01 while the test error touches its minimum (0.109) around 20 rounds and then climbs back (0.139 at 300 rounds). It is the sign of the overfitting of boosting: more rounds = more complexity, and the later rounds learn the noise of the residuals.
- **After 100 rounds small differences with scikit-learn appear** (up to about 0.1 in absolute value): scikit-learn converts the features to `float32` and resolves ties between splits differently, and the differences accumulate. It is not a mistake: two correct implementations of the same algorithm can diverge in the numerical details.
- Compare with the forest (M8): the forest averages **deep** trees and never climbs back; boosting sums **small** trees and can climb back. That is why the number of rounds is a hyperparameter to choose with validation.

## Common pitfall

Increasing the rounds as long as the training error drops. In boosting the training error always drops (there is always something left to learn in the residuals): the only measure of the "right point" is the error on unseen data.

## Interview question

> *Click on a question to show its answer.*

<details><summary>How does gradient boosting work with the squared loss?</summary>

It starts from a constant prediction (the mean of the target); at each round it computes the residuals of the current model, trains a small tree to predict the residuals and adds its prediction multiplied by a learning rate. With other losses the residuals become the negative gradient of the loss.
The training error always drops, so the number of rounds is chosen with a validation set (early stopping).

</details>